In [1]:
import colapy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import tempfile
import os

from tqdm import tqdm


urqmd_events = list[colapy.EventData]()


class MemoryWriter(colapy.WriterBase):
    def __init__(self, **kwargs: dict[str, str]) -> None:
        urqmd_events.clear()

    def __call__(self, event_data: colapy.EventData) -> None:
        urqmd_events.append(event_data)


CONFIG = '''
<?xml version="1.0" encoding="UTF-8" ?>
<program>
    <generator name="URQMDGenerator"
        pro="197 79"
        tar="197 79"
        nev="1"
        imp="5."
        elb="100."
        tim="200 200"
        generated_config_file="input_file"/>
    <writer name="PythonWriter" class="MemoryWriter"/>
</program>
'''

N = 3

with tempfile.NamedTemporaryFile(mode='w', suffix='.xml', delete_on_close=False) as tmp:
    tmp.name
    tmp.write(CONFIG)
    tmp.close()

    print('Initializing Run Manager...')
    rm = colapy.RunManager().load_module('COLA-Py').load_module('COLA_UrQMD').load_config(tmp.name)
    print('Running...')
    for _ in tqdm(range(N)):
        rm.run(1)

    os.remove('input_file')


Initializing Run Manager...
Parsing XML file:
filter name: URQMDGenerator
params:
generated_config_file: input_file
elb: 100.
tar: 197 79
imp: 5.
nev: 1
tim: 200 200
pro: 197 79
 #############################################################
 ##                                                         ##
 ## UrQMD 4.0        University of Frankfurt                ##
 ##                  http://urqmd.org                       ##
 ##                  bleicher@itp.uni-frankfurt.de          ##
 #############################################################
 ##                                                         ##
 ##     Please cite when using this model:                  ##
 ##     S.A.Bass et al., Prog.Part.Nucl.Phys. 41 (1998) 225 ##
 ##     M.Bleicher et al., J.Phys. G25  (1999) 1859         ##
 ##                                                         ##
 #############################################################
 ##     UrQMD uses Pythia6.409 by T. Sjorstrand             ##
 ##

  0%|          | 0/3 [00:00<?, ?it/s]

 (info) dsigma: calculating constants for ang. dist.
 (info) dsigma: calculation finished


100%|██████████| 3/3 [00:10<00:00,  3.56s/it]


 Advisory warning: maximum violated by  1.090D+00 in event       1
 XSEC(96,1) increased to  1.806D+03


In [3]:
# import pickle

# def to_dict(obj: object) -> object:
#     if isinstance(obj, (int, str, float)):
#         return obj

#     if isinstance(obj, list):
#         return [to_dict(el) for el in obj]

#     if isinstance(obj, dict):
#         return {
#             to_dict(k): to_dict(v) for k, v in obj.items()
#         }

#     return {
#         el: to_dict(getattr(obj, el))
#         for el in dir(obj)
#         if not el.startswith('_')
#     }


# with open('out.pkl', 'wb') as f:
#     pickle.dump(list(map(to_dict, urqmd_events)), f)


In [7]:
def to_phase_space(particle: colapy.Particle) -> tuple[float, ...]:
    return (
        particle.position.t,
        particle.position.x,
        particle.position.y,
        particle.position.z,
        particle.momentum.e,
        particle.momentum.x,
        particle.momentum.y,
        particle.momentum.z,
    )


def get_points(particles: list[colapy.Particle]) -> np.ndarray:
    result = []
    for p in particles:
        if p.pdg_code not in [2212, 2112]:
            continue

        result.append(to_phase_space(p))

    return np.asarray(result, dtype=np.float64)


points = get_points(urqmd_events[0].particles)
len(urqmd_events[0].particles), len(points)

(1760, 355)

In [8]:
import typing as tp


def particle_to_dict(p: colapy.Particle) -> dict[str, tp.Any]:
    return {
        'pdg_code': p.pdg_code,
        'class': str(p.p_class),
        't': p.position.t,
        'x': p.position.x,
        'y': p.position.y,
        'z': p.position.z,
        'e': p.momentum.e,
        'px': p.momentum.x,
        'py': p.momentum.y,
        'pz': p.momentum.z,
    }


df = pd.DataFrame(map(particle_to_dict, urqmd_events[0].particles))
df

,pdg_code,class,t,x,y,z,e,px,py,pz
0,3112,ParticleClass.PRODUCED,200.0,85.652411,18.746375,-138.491374,2211.309895,982.398529,221.083111,-1566.857609
1,2112,ParticleClass.PRODUCED,200.0,17.515152,19.201400,195.492871,6707.530390,409.565762,561.664081,6605.142635
2,2212,ParticleClass.ELASTIC_A,200.0,15.314215,0.456483,196.882279,7684.312279,406.053655,-36.106382,7615.945635
3,2212,ParticleClass.PRODUCED,200.0,5.919208,-3.224957,194.848615,5118.493509,14.700361,-154.546020,5029.416585
4,2212,ParticleClass.PRODUCED,200.0,29.563125,104.170604,141.004875,2155.662119,340.572938,1117.590016,1549.850882
...,...,...,...,...,...,...,...,...,...,...
1755,-311,ParticleClass.PRODUCED,200.0,-67.959353,10.691519,-169.806689,1516.261087,-568.119283,50.809790,-1315.169396
1756,-211,ParticleClass.PRODUCED,200.0,5.493085,48.675615,-177.535454,394.621759,31.513512,-31.770364,-366.987569
1757,211,ParticleClass.PRODUCED,200.0,5.493085,48.675615,-177.535454,236.699189,60.104181,25.822144,-180.840293
1758,-211,ParticleClass.PRODUCED,200.0,41.307163,29.414367,147.955640,345.883464,-56.048090,50.721973,308.021532


In [9]:
properties = {
    111: (0, 0), 443: (0, 0), 211: (0, 1), -211: (0, -1),
    2212: (1, 1), 2112: (1, 0), 311: (0, 0), 321: (0, 1),
    221: (0, 0), -321: (0, -1), -311: (0, 0), 3122: (1, 0),
    3222: (1, 1), 3112: (1, -1), 3212: (1, 0), -2212: (-1, -1),
    -3222: (-1, -1), 3312: (1, -1), -3212: (-1, 0),
    3322: (1, 0), -3322: (-1, 0), -2112: (-1, 0),
    -3312: (-1, 1), -3122: (-1, 0),
}

arr = []
for event in urqmd_events:
    total_B = 0
    total_Q = 0
    for p in event.particles:
        B, Q = properties[p.pdg_code]
        total_B += B
        total_Q += Q

    arr.append((total_B, total_Q))

np.mean(arr, axis=0), 197 * 2, 79 * 2

(array([394., 158.]), 394, 158)

In [10]:
import plotly.graph_objects as go
import numpy as np


points = get_points(urqmd_events[0].particles)

fig = go.Figure(data=[go.Scatter3d(
    x=points[:, 1], y=points[:, 2], z=points[:, 3],
    mode='markers',
    marker=dict(
        size=3,
        color='blue',
        colorscale='Viridis',
        opacity=0.8
    ),
    hoverinfo='text'
)])

fig.update_layout(
    scene=dict(
        xaxis_title='X (fm)',
        yaxis_title='Y (fm)',
        zaxis_title='Z (fm)'
    ),
    title='Nucleon Positions after Clustering'
)
fig.show()

In [ ]:
import math

from masstable import Table

# AME2012all: install with ``pip install masstable``
ame = Table("AME2012all")

M_P = 938.2723
M_N = 939.5656

# Try in order: experimental masses, then DUZU / FRDM95 (``masstable``), then liquid drop.
_MASS_TABLE_NAMES: tuple[str, ...] = ["AME2012all"]
_table_cache: dict[str, Table] = {}


def _table(name: str) -> Table:
    if name not in _table_cache:
        _table_cache[name] = Table(name)
    return _table_cache[name]


def binding_energy_liquid_drop(a: int, z: int) -> float:
    if a <= 0 or z < 0 or z > a:
        raise ValueError(f"invalid (A, Z)=({a}, {z})")

    n = a - z

    if a == 1:
        return 0.0

    a_v, a_s, a_c, a_a = 15.75, 17.8, 0.711, 23.7
    vol = a_v * a
    surf = a_s * (a ** (2.0 / 3.0))
    coul = a_c * z * (z - 1.0) / (a ** (1.0 / 3.0))
    asym = a_a * ((n - z) ** 2) / a

    if a % 2 == 1:
        delta = 0.0
    elif z % 2 == 0:
        delta = 12.0 / math.sqrt(float(a))
    else:
        delta = -12.0 / math.sqrt(float(a))

    return float(vol - surf - coul - asym + delta)


def binding_energy_from_tables(a: int, z: int) -> float | None:
    """Binding energy in MeV from first ``masstable`` table that contains (Z, N), or None."""
    n = a - z
    key = (z, n)
    for name in _MASS_TABLE_NAMES:
        t = _table(name)
        if key in t.binding_energy.df.index:
            return float(t.binding_energy.df.loc[key])
    return None


def get_mass(a: int, z: int) -> float:
    n = a - z
    b = binding_energy_from_tables(a, z)
    if b is None:
        b = binding_energy_liquid_drop(a, z)

    return z * M_P + n * M_N - b


def binding_energy(a: int, z: int) -> float:
    """Total binding energy in MeV (positive = bound); same fallbacks as ``get_mass``."""
    b = binding_energy_from_tables(a, z)
    if b is not None:
        return b
    return binding_energy_liquid_drop(a, z)


get_mass(4, 2)


/Users/artyasen/.pyenv/versions/3.12.4/lib/python3.12/site-packages/masstable/masstable.py:89: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(filename, header=0, delim_whitespace=True, index_col=[0, 1])[
/Users/artyasen/.pyenv/versions/3.12.4/lib/python3.12/site-packages/masstable/masstable.py:89: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(filename, header=0, delim_whitespace=True, index_col=[0, 1])[


3727.379032

In [12]:
pdg_to_az = {
    2212: (1, 1),
    2112: (1, 0),
}

def lorentz_len(vec: np.ndarray) -> np.ndarray:
    assert vec.shape[-1] == 4

    return np.sqrt(vec[..., 0] ** 2 - np.sum(vec[..., 1:] ** 2, axis=-1))


def make_group(group: list[colapy.Particle]) -> tuple[np.ndarray, np.ndarray, float]:
    points = np.asarray(list(map(to_phase_space, group)))
    points_moms = points[:, 4:]
    group_pos = np.mean(points[:, :4], axis=0)
    group_mom = np.sum(points_moms, axis=0)

    group_a, group_z = 0, 0
    for p in group:
        particle_a, particle_z = pdg_to_az[p.pdg_code]
        group_a += particle_a
        group_z += particle_z

    m_inv = float(lorentz_len(group_mom))
    m_gs = get_mass(group_a, group_z)
    delta = m_inv - m_gs

    return (
        group_pos,
        group_mom,
        delta,
    )


In [13]:
nucleus = list(filter(lambda x: x.pdg_code in pdg_to_az, urqmd_events[0].particles))

In [22]:
make_group([ nucleus[id] for id in np.random.choice(len(nucleus), 4)])

(array([200.        ,  18.39535056,   8.35084197,  -7.88801518]),
 array([25655.45403512,  1305.48938166,   435.68012952, -5245.72713361]),
 21325.93326690595)